# Module 09 — Seeing What the Filters See

A CNN's first layer is a set of small **kernels** (filters). Each one lights up
where its pattern appears in the image. This notebook makes that visible: we
apply a bank of filters to a digit and look at each filter's "activation map."
You'll see edge detectors firing on edges — the same thing a trained CNN learns
to do, except it discovers the kernels itself.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
image = digits.images[7]        # an 8x8 handwritten digit
plt.imshow(image, cmap="gray"); plt.title("input digit"); plt.axis("off"); plt.show()

In [ ]:
def convolve2d(image, kernel):
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    padded = np.pad(image, ((ph, ph), (pw, pw)), mode="edge")
    out = np.zeros_like(image, dtype=float)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            out[i, j] = np.sum(padded[i:i+kh, j:j+kw] * kernel)
    return out

## A bank of filters (feature detectors)
Vertical edge, horizontal edge, two diagonals, and a blob detector. In a real
CNN these weights would be LEARNED by backprop; here we set them so you can see
what "a filter detects" means.

In [ ]:
filters = {
    "vertical edge":   np.array([[-1,0,1],[-1,0,1],[-1,0,1]]),
    "horizontal edge": np.array([[-1,-1,-1],[0,0,0],[1,1,1]]),
    "diagonal \\":     np.array([[2,-1,-1],[-1,2,-1],[-1,-1,2]]),
    "diagonal /":      np.array([[-1,-1,2],[-1,2,-1],[2,-1,-1]]),
    "blob/center":     np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]]),
}

fig, axes = plt.subplots(2, len(filters), figsize=(3*len(filters), 6))
for col, (name, k) in enumerate(filters.items()):
    axes[0, col].imshow(k, cmap="RdBu"); axes[0, col].set_title(name, fontsize=9)
    axes[0, col].axis("off")
    act = np.abs(convolve2d(image, k))
    axes[1, col].imshow(act, cmap="viridis"); axes[1, col].set_title("activation")
    axes[1, col].axis("off")
plt.tight_layout(); plt.show()
print("Top row: the filter weights (red/blue). Bottom row: where it fired on the digit.")

## Reading the activations
Each filter's activation map is bright exactly where its pattern occurs:
the vertical-edge filter lights up vertical strokes, the diagonal filters light
up slanted strokes, and so on. A CNN stacks many such maps; a later layer sees
"there's a vertical edge here AND a curve there" and concludes "this is a 7."

## Optional: visualize a REAL learned filter (needs torch)
If you trained `cnn_classifier.py`, its `conv1.weight` holds the kernels the
network learned. Uncomment to peek — they often resemble the edge/blob detectors
above, discovered from data alone.

In [ ]:
try:
    import torch
    # from cnn_classifier import SmallCNN   # if you saved a trained model
    print("torch available — you could load conv1.weight and imshow each 3x3 kernel.")
    print("They typically look like fuzzy edge/curve detectors: learned, not designed.")
except ImportError:
    print("torch not installed — the hand-set filters above already show the idea.")

## Security note: adversarial perturbations
Because a CNN's decision is a smooth function of pixel values, a tiny,
carefully-chosen change to many pixels (imperceptible to you) can shift the
activation maps just enough to flip the prediction — an **adversarial example**.
Understanding filters is step one to understanding why these attacks exist and
how detectors/defenses for them work.

**Next:** back to `tutorial.html`, then `project.md`.